# Step 1: Download/Pre-process SFS forecast data
Lindsay Fitzpatrick
ljob@umich.edu

**Initial Version:** 09/14/2026  
**Last Updated:**

This script automates the download and preprocessing of **Seasonal Forecast System (SFS) Beta** forecast data for the Great Lakes region. The data is sourced from ZARR files on **AWS**.

### Key Functionality
- Downloads SFS forecast files for specified dates and forecast hours.
- Processes the data to calculate key atmospheric metrics:
  - Total precipitation
  - Evaporation
  - 2-meter air temperature (averaged separately over lake and land surfaces)
- Applies a geographic mask to isolate values over lake and land areas using `GL_mask.nc`.
- Saves the processed results into a local **SQLite database** (`sfs_forecast_data.db`), either by appending to an existing file or creating a new one.

### Required Input Files
- **`GL_mask.nc`**  
  A NetCDF file containing geographic masks to distinguish lake and land grid cells in the Great Lakes region.

- **`cfs_forecast_data.db`** *(optional)*  
  If provided, the script will append new forecast data to this database; otherwise, it will create a new database with the appropriate schema.

This script forms the first step in the CNBS forecasting pipeline, ensuring that clean, lake-specific forecast variables are prepared for use in downstream modeling and prediction.

In [1]:
import os
import sys
from datetime import datetime
import pandas as pd
import xarray as xr

# Add the path to the src directory (two levels up)
sys.path.append(os.path.abspath('../../'))
from src.data_downloader import CFSDownloader
#from src.data_processor import CFSProcessor
from src.database_utils import Database

## User Inputs
### Configuration: File Paths and Processing Options

This section defines key file paths and user-configurable options for downloading and processing CFS forecast data:

- **`local_path`**: Base directory where the repository is cloned and data folders are located.  
- **`download_dir`**: Directory where raw CFS GRIB2 files will be downloaded and temporarily stored.  
- **`input_dir`**: Directory containing required input files such as masks and scalers.  
- **`mask_file`**: Path to the Great Lakes mask file (`GL_mask.nc`) used for separating lake and land data.  
- **`database`**: Path to the SQLite database where processed CFS forecast data is stored.  

#### Data Source and Processing Control  
- **`source`**: Specifies the data source, either `'aws'` or `'ncei'`.  
- **`download_cfs`**: Toggle to enable or disable downloading new CFS data (`'yes'` or `'no'`).  
- **`process_cfs`**: Toggle to enable or disable processing of downloaded CFS data (`'yes'` or `'no'`).  
- **`delete_files`**: Option to delete raw GRIB2 files after processing to save storage (`'yes'` or `'no'`).  

#### Date Range Configuration  
- **`auto`**: When set to `'yes'`, automatically updates the database by detecting the last processed date and downloading data up to yesterday’s date.  
- If **`auto`** is `'no'`, specify manual start and end dates for data download and processing:  
  - **`start_date`**: Starting date for data retrieval (format: MM-DD-YYYY).  
  - **`end_date`**: Ending date for data retrieval (format: MM-DD-YYYY).  

These settings allow flexible control over data acquisition and processing, supporting both automated updates and manual testing or reprocessing.

In [8]:
# Directory where the repository is cloned
local_path = '/Users/ljob/Desktop/'

# Path to data directory
input_dir = local_path + 'cnbs-predictor/data/'

# Path the GL mask file
mask_file = input_dir + 'input/GL_mask_sfs.nc'

# Path to the SFS forecast database
database = input_dir + 'sfs/sfs_forecast_data.db'
table = 'sfs_forecast_data'

# Auto mode will automatically open the existing database, pull the last entered date to determine the start date, 
# and set the end date to yesterday, making the database 'up-to-date'. If 'no', you can manually enter a start and 
# end date (ideal for testing or if you need to redownload/reprocess specific time frames).
auto = 'no'

# Specify the start and end dates if auto mode above is set to 'no' (Format: MM-YYYY)
start_date = '02-2026'
end_date = '09-2026'

### Preset Variables

This section defines key preset variables used throughout the SFS data download and processing workflow:

- **`mask_variables`**: Names of the lake and land regions used for applying the spatial mask.  
  Each entry corresponds to a specific Great Lake (e.g., Erie, Ontario, Michigan-Huron, Superior), with distinctions for lake surface and surrounding land areas.

- **`bucket_name`**: AWS S3 bucket name (`noaa-cfs-pds`) used to access the public NOAA CFS forecast data.

These presets standardize the data sources, time intervals, and spatial definitions used in the forecast extraction and preprocessing steps.

In [3]:
# Define mask variables
mask_variables = ['erie_lake', 'erie_land',
                  'ontario_lake', 'ontario_land',
                  'michigan-huron_lake', 'michigan-huron_land',
                  'superior_lake', 'superior_land']

# AWS bucket name to locate the SFS forecast
bucket_name = 'noaa-oar-sfsdev-pds'

# List of variables to include in the dataset
variables = [
    "pratesfc",
    "tmp2m",
    "avg_slhtfsfc",
]

## Begin Script

Create a database handler.

In [4]:
db = Database(database, table)

### Determine Date Range for Forecast Download and Processing

This section defines the date range over which the script will operate, based on the user's selected mode:

- When **`auto = 'yes'`**:  
  The script automatically checks the `cfs_forecast_data.db` database for the most recent recorded CFS run.  
  - The **start date** is set to the day after the last recorded run.  
  - The **end date** is set to **yesterday**, ensuring the database stays up to date with the latest available forecasts.

- When **`auto = 'no'`**:  
  The user manually specifies both the start and end dates.  
  This is especially useful for testing or initializing a new database with historical forecasts.

Before proceeding, the script prints a message confirming the selected date range, for example:  
`Starting from: 05-22-2025 00Z and continuing through: 06-10-2025 18Z`

This ensures transparency and gives the user a final opportunity to confirm the configured date range before data download and processing begins.

In [9]:
#start_date, end_date, date_array = db.get_date_range(auto, start_date, end_date)

# ---------------------------------------------------------
# Create monthly date range
# ---------------------------------------------------------

date_array = pd.date_range(
    start=datetime.strptime(start_date, "%m-%Y"),
    end=datetime.strptime(end_date, "%m-%Y"),
    freq="MS",
)

print(date_array)

DatetimeIndex(['2026-02-01', '2026-03-01', '2026-04-01', '2026-05-01',
               '2026-06-01', '2026-07-01', '2026-08-01', '2026-09-01'],
              dtype='datetime64[us]', freq='MS')


### Loop Through Forecast Dates: Download, Process, and Store Data

This section iterates over each date in the user-defined `date_array` to handle CFS forecast data for the Great Lakes using dedicated modules and functions.

For each date:

1. **Prints progress** to the console (e.g., "Beginning Files for 2025-06-10").

2. **Download CFS forecast files** (`download_cfs = 'yes'`):  
   - Uses the `CFSDownloader` module to fetch forecast data.  
   - The `download()` function automatically constructs the correct file paths based on the chosen source (`aws` or `ncei`) and downloads all relevant GRIB2 files for that date.

3. **Process downloaded GRIB2 files** (`process_cfs = 'yes'`):  
   - The `CFSProcessor` module reads GRIB2 files from the daily subdirectory.  
   - `process_files()` calculates atmospheric metrics (e.g., precipitation, evaporation, temperature) and applies spatial masks.  
   - Results are saved to the configured SQLite database (`cfs_forecast_data.db`) under the `cfs_forecast_data` table.

4. **Optional cleanup** (`delete_files = 'yes'`):  
   - Deletes the local GRIB2 directory for that date to conserve storage space.  
   - If the directory cannot be deleted, a warning is printed and processing continues.

After all dates are processed, the script prints:  
**"Process Complete"**

This modular loop ensures a clear, maintainable pipeline for downloading, processing, and storing CFS forecast data efficiently using the new `CFSDownloader` and `CFSProcessor` classes.

In [ ]:
# ---------------------------------------------------------
# Loop through monthly SFS forecasts
# ---------------------------------------------------------

for date in date_array:

    yyyy_mm = date.strftime("%Y%m")
    print(yyyy_mm.type)
    
    zarr_path = (
        f"s3://{bucket_name}/"
        f"experiments/beta1/"
        f"forecast/{yyyy_mm}/"
        f"atm_monthly.zarr"
    )

    print()
    print(f"Reading SFS forecast: {yyyy_mm}")

    try:

        ds = xr.open_zarr(
            zarr_path,
            storage_options={"anon": True},
        )[variables]

        print(ds)

        init_timestamp = pd.Timestamp(yyyy_mm + "01")
        seconds_in_month = init_timestamp.days_in_month * 86400

        print()
        print("=" * 70)
        print(
            f"Processing initialization: "
            f"{init_timestamp:%Y-%m-%d}"
        )
        print("=" * 70)

        for member in variable.member.values:

            member = int(member)

            print()
            print(f"  Member: {member}")
            for lead in variable.lead.values:

                lead = int(lead)
                valid_time = (
                    init_timestamp
                    + pd.DateOffset(months=lead)
                )


                print(
                    f"    Lead {lead:02d} "
                    f"-> {valid_time:%Y-%m}"
                )

                # ===================================================
                # Select temperature field
                # ===================================================

                tmp = variable.sel(
                    init=init_time,
                    member=member,
                    lead=lead,
                )

                # ===================================================
                # Cut temperature field to mask extent
                # ===================================================

                tmp_cut = tmp.sel(
                    lat=slice(
                        mask_lat.min(),
                        mask_lat.max(),
                    ),
                    lon=slice(
                        mask_lon.min(),
                        mask_lon.max(),
                    ),
                )

                tmp_values = tmp_cut.values

                # ===================================================
                # Loop through lake/land masks
                # ===================================================

                for mask_var in mask_variables:

                    lake, surface_type = mask_var.rsplit(
                        "_",
                        1,
                    )

                    # ------------------------------------------------
                    # Get fractional mask
                    # ------------------------------------------------

                    mask = mask_ds[mask_var].values

                    # ------------------------------------------------
                    # Make sure mask and temperature have compatible
                    # dimensions.
                    # ------------------------------------------------

                    if mask.shape != tmp_values.shape:

                        raise ValueError(
                            f"Mask shape does not match temperature "
                            f"shape for {mask_var}: "
                            f"mask={mask.shape}, "
                            f"temperature={tmp_values.shape}"
                        )
                    # LATENT HEAT FLUX
                    #tmp_avg = (
                    #    np.nansum(tmp_values * mask * area)
                    #    / np.nansum(mask * area)
                    # )
                    
                    # NEEDED FOR PRECIPITATION RATE CONVERSION FROM RATE TO TOTAL AMOUNT OVER THE MONTH
                    weighted_tmp_rate = (
                        np.nansum(tmp_values * mask * area)
                        / np.nansum(mask * area)   
                    )
                    
                    tmp_avg = weighted_tmp_rate * seconds_in_month
                    # ------------------------------------------------

                    # START OF TEMPERATURE AVERAGING

                    # ------------------------------------------------
                    # Identify valid cells.
                    #
                    # Exclude:
                    #
                    #   - NaN mask cells
                    #   - NaN temperature cells
                    # ------------------------------------------------

                    #valid = (
                    #    np.isfinite(mask)
                    #    & np.isfinite(tmp_values)
                    #)

                    # ------------------------------------------------
                    # Extract valid values and weights
                    # ------------------------------------------------

                    #values = tmp_values[valid]
                    #weights = mask[valid]
                    
                    # ------------------------------------------------
                    # Calculate weighted mean
                    #
                    # Fractional masks are handled here:
                    #
                    # 1.00 = full cell
                    # 0.75 = 75% of cell
                    # 0.50 = 50% of cell
                    # 0.25 = 25% of cell
                    #
                    # NaN cells are excluded.
                    # ------------------------------------------------

                    #if (
                    #    len(values) == 0
                    #    or np.sum(weights) == 0
                    #):

                    #tmp_avg = np.nan

                    #else:

                    #    tmp_avg = np.average(
                    #        values,
                    #        weights=weights,
                    #    )

                    # END OF TEMPERATURE AVERAGING


                    # ------------------------------------------------
                    # Insert result into database
                    # ------------------------------------------------

                    cursor.execute(
                        """
                        INSERT INTO reforecast_data (
                            init_time,
                            member,
                            lead,
                            valid_time,
                            lake,
                            surface_type,
                            component,
                            value
                        )
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
                        """,
                        (
                            init_timestamp.strftime(
                                "%Y-%m-%d"
                            ),
                            member,
                            lead,
                            valid_time.strftime(
                                "%Y-%m-%d"
                            ),
                            lake,
                            surface_type,
                            long_name,
                            (
                                float(tmp_avg)
                                if np.isfinite(tmp_avg)
                                else None
                            ),
                        ),
                    )


            # -------------------------------------------------------
            # Commit after each ensemble member
            #
            # This prevents a large number of uncommitted database
            # transactions from accumulating.
            # -------------------------------------------------------

            conn.commit()

    except Exception as e:

        print()
        print("=" * 70)
        print(f"    Error reading {yyyy_mm}: {e}")
        print("    Skipping this month and continuing on...")
        print("=" * 70)
        print()

        continue

print("***** Process Complete *****")

IndentationError: unexpected indent (349116191.py, line 256)